# Path B AOM Design

経路B用のレンズ配置を検討する notebook です。

- 初期ビームは `Beam_Gaussian_Fitting.ipynb` の Cell 7 の結果を使う
- レンズ配置の考え方は `OptimizeDivergenceAngle.ipynb` をベースにする
- AOM 開口、カップラー位置、SM ファイバーの MFD を使って候補を評価する
- 非点収差は最終候補比較のための参考値として出す


In [ ]:
import math
from dataclasses import dataclass

import numpy as np


## Initial Beam Parameters

`Beam_Gaussian_Fitting.ipynb` の Cell 7 の値をそのまま使います。


In [ ]:
LAM = 852.3e-9  # [m]
W0_X = 162.6e-6  # [m] horizontal
Z0_X = -101.8e-3  # [m]
W0_Y = 334.7e-6  # [m] vertical
Z0_Y = -24.7e-3  # [m]


## AOM Specifications

今回使う AOM の規格を入れます。


In [ ]:
AOM_WAVELENGTH_MIN = 780e-9  # [m]
AOM_WAVELENGTH_MAX = 850e-9  # [m]
AOM_OPERATING_FREQUENCY = 200e6  # [Hz]
AOM_ACTIVE_APERTURE = 0.32e-3  # [m]
AOM_RISE_FALL_TIME = 10e-9  # [s]


## Fiber Target

Single-mode fiber の MFD から、目標発散角を作ります。


In [ ]:
SM_MFD = 5.3e-6  # [m]
SM_TARGET_DIVERGENCE = LAM / (math.pi * (SM_MFD / 2.0))  # [rad]

print(f"SM target divergence = {SM_TARGET_DIVERGENCE:.5f} rad")


## Utility Functions

q パラメータ、自由空間伝搬、レンズ通過、各位置でのビーム径評価に使う関数です。


In [ ]:
@dataclass
class AxisState:
    q: complex
    waist_position: float
    rayleigh_length: float
    waist_radius: float


@dataclass
class PlaneReport:
    z_mm: float
    radius_x_um: float
    radius_y_um: float
    diameter_x_um: float
    diameter_y_um: float
    max_diameter_um: float


def q_from_waist(w0, z0):
    z_r = math.pi * w0**2 / LAM
    return -z0 + 1j * z_r


def gaussian_beam_radius_from_q(q, dz):
    q_plane = q + dz
    inv_q = 1.0 / q_plane
    imag_inv_q = inv_q.imag
    if imag_inv_q >= 0:
        raise ValueError("Invalid q parameter: expected negative imaginary part for 1/q.")
    return math.sqrt(-LAM / (math.pi * imag_inv_q))


def propagate_free_space(q, distance):
    return q + distance


def propagate_lens(q, focal_length):
    return 1.0 / ((1.0 / q) - (1.0 / focal_length))


def state_from_q(q, plane_z):
    waist_position = plane_z - q.real
    rayleigh_length = q.imag
    waist_radius = math.sqrt(LAM * rayleigh_length / math.pi)
    return AxisState(
        q=q,
        waist_position=waist_position,
        rayleigh_length=rayleigh_length,
        waist_radius=waist_radius,
    )


def plane_report(qx_ref, qy_ref, ref_z, plane_z):
    dz = plane_z - ref_z
    radius_x = gaussian_beam_radius_from_q(qx_ref, dz)
    radius_y = gaussian_beam_radius_from_q(qy_ref, dz)
    return PlaneReport(
        z_mm=plane_z * 1e3,
        radius_x_um=radius_x * 1e6,
        radius_y_um=radius_y * 1e6,
        diameter_x_um=2.0 * radius_x * 1e6,
        diameter_y_um=2.0 * radius_y * 1e6,
        max_diameter_um=max(2.0 * radius_x, 2.0 * radius_y) * 1e6,
    )


## Main Evaluation Function

2 枚レンズ系を通したあと、AOM 面・カップラー面・カップラーレンズ後の発散角を評価します。


In [ ]:
def evaluate_path_b(d1, f1, d2, f2, aom_z, coupler_z, f3):
    q0_x = q_from_waist(W0_X, Z0_X)
    q0_y = q_from_waist(W0_Y, Z0_Y)

    q1_x = propagate_lens(propagate_free_space(q0_x, d1), f1)
    q1_y = propagate_lens(propagate_free_space(q0_y, d1), f1)
    lens1_z = d1

    q2_x = propagate_lens(propagate_free_space(q1_x, d2), f2)
    q2_y = propagate_lens(propagate_free_space(q1_y, d2), f2)
    lens2_z = d1 + d2

    lens2_state_x = state_from_q(q2_x, lens2_z)
    lens2_state_y = state_from_q(q2_y, lens2_z)

    aom_plane = plane_report(q2_x, q2_y, lens2_z, aom_z)
    coupler_plane = plane_report(q2_x, q2_y, lens2_z, coupler_z)

    aom_pass = (aom_plane.max_diameter_um * 1e-6) <= AOM_ACTIVE_APERTURE
    aom_fill_ratio = (aom_plane.max_diameter_um * 1e-6) / AOM_ACTIVE_APERTURE

    q_at_coupler_x = propagate_free_space(q2_x, coupler_z - lens2_z)
    q_at_coupler_y = propagate_free_space(q2_y, coupler_z - lens2_z)

    q_after_coupler_lens_x = propagate_lens(q_at_coupler_x, f3)
    q_after_coupler_lens_y = propagate_lens(q_at_coupler_y, f3)

    coupler_state_x = state_from_q(q_after_coupler_lens_x, coupler_z)
    coupler_state_y = state_from_q(q_after_coupler_lens_y, coupler_z)

    theta_x = LAM / (math.pi * coupler_state_x.waist_radius)
    theta_y = LAM / (math.pi * coupler_state_y.waist_radius)

    return {
        "lens1_z_mm": lens1_z * 1e3,
        "lens2_z_mm": lens2_z * 1e3,
        "lens2_state_x": lens2_state_x,
        "lens2_state_y": lens2_state_y,
        "aom_plane": aom_plane,
        "coupler_plane": coupler_plane,
        "aom_pass": aom_pass,
        "aom_fill_ratio": aom_fill_ratio,
        "theta_x": theta_x,
        "theta_y": theta_y,
        "theta_target": SM_TARGET_DIVERGENCE,
        "theta_error": max(abs(theta_x - SM_TARGET_DIVERGENCE), abs(theta_y - SM_TARGET_DIVERGENCE)),
        "astigmatism_after_lens2_mm": abs(lens2_state_x.waist_position - lens2_state_y.waist_position) * 1e3,
        "astigmatism_after_coupler_mm": abs(coupler_state_x.waist_position - coupler_state_y.waist_position) * 1e3,
    }


def search_coupler_positions(d1, f1, d2, f2, aom_z, f3, start_z, stop_z, num_points):
    candidates = []
    for coupler_z in np.linspace(start_z, stop_z, num_points):
        result = evaluate_path_b(d1, f1, d2, f2, aom_z, float(coupler_z), f3)
        if result["aom_pass"]:
            candidates.append(result)
    candidates.sort(
        key=lambda item: (
            item["theta_error"],
            item["astigmatism_after_coupler_mm"],
            abs(item["coupler_plane"].diameter_x_um - item["coupler_plane"].diameter_y_um),
        )
    )
    return candidates


## Print Helpers

結果を見やすく出すための関数です。


In [ ]:
def print_plane_report(name, report):
    print(f"[{name}] z = {report.z_mm:.1f} mm")
    print(f"  radius_x   = {report.radius_x_um:.1f} um")
    print(f"  radius_y   = {report.radius_y_um:.1f} um")
    print(f"  diameter_x = {report.diameter_x_um:.1f} um")
    print(f"  diameter_y = {report.diameter_y_um:.1f} um")
    print(f"  max_diam   = {report.max_diameter_um:.1f} um")


def print_result(title, result):
    print(f"=== {title} ===")
    print(f"lens1 position = {result['lens1_z_mm']:.1f} mm")
    print(f"lens2 position = {result['lens2_z_mm']:.1f} mm")
    print_plane_report("AOM plane", result["aom_plane"])
    print(f"AOM aperture check = {'PASS' if result['aom_pass'] else 'FAIL'}")
    print(f"AOM fill ratio     = {result['aom_fill_ratio']:.3f}")
    print_plane_report("Coupler plane", result["coupler_plane"])
    print(f"theta_x            = {result['theta_x']:.5f} rad")
    print(f"theta_y            = {result['theta_y']:.5f} rad")
    print(f"theta_target(MFD)  = {result['theta_target']:.5f} rad")
    print(f"theta_error(max)   = {result['theta_error']:.5f} rad")
    print(f"astig after lens2  = {result['astigmatism_after_lens2_mm']:.1f} mm")
    print(f"astig after coupler= {result['astigmatism_after_coupler_mm']:.1f} mm")
    print()


## Candidate Evaluation

`OptimizeDivergenceAngle.ipynb` で使っていた代表値をまず評価します。


In [ ]:
candidate_1 = evaluate_path_b(
    d1=350e-3,
    f1=150e-3,
    d2=350e-3,
    f2=100e-3,
    aom_z=575e-3,
    coupler_z=950e-3,
    f3=7.5e-3,
)

candidate_2 = evaluate_path_b(
    d1=362.5e-3,
    f1=150e-3,
    d2=350e-3,
    f2=100e-3,
    aom_z=575e-3,
    coupler_z=950e-3,
    f3=7.5e-3,
)

print_result("Candidate 1 (d1=350 mm, d2=350 mm, AOM=575 mm, coupler=950 mm)", candidate_1)
print_result("Candidate 2 (d1=362.5 mm, d2=350 mm, AOM=575 mm, coupler=950 mm)", candidate_2)


## Coupler Search

AOM 面で開口を満たすことを条件に、カップラー位置候補を探索します。


In [ ]:
search = search_coupler_positions(
    d1=362.5e-3,
    f1=150e-3,
    d2=350e-3,
    f2=100e-3,
    aom_z=575e-3,
    f3=7.5e-3,
    start_z=750e-3,
    stop_z=1200e-3,
    num_points=181,
)

print("=== Top Coupler Candidates ===")
for idx, result in enumerate(search[:10], start=1):
    coupler_z = result["coupler_plane"].z_mm
    print(
        f"{idx}. coupler={coupler_z:.1f} mm, "
        f"theta_error={result['theta_error']:.5f} rad, "
        f"astig={result['astigmatism_after_coupler_mm']:.1f} mm, "
        f"AOM_fill={result['aom_fill_ratio']:.3f}"
    )
